# Motor de Web Scraping — Ofertas Financieras Competencia Automoción
**Honda Financial Services | TFM BSM Barcelona**

Extrae ofertas de financiación (TIN, TAE, comisión de apertura, cuota, plazo, etc.) de webs de competidores usando scraping + LLM (Claude).

## 1. Instalación de dependencias (ejecutar solo en Colab)

In [ ]:
# Ejecuta esta celda la primera vez en Google Colab (tarda ~1 minuto)
!pip install -q requests beautifulsoup4 selenium pandas webdriver-manager openai
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y -q ./google-chrome-stable_current_amd64.deb
print("Dependencias instaladas correctamente")

## 2. Imports y configuración

In [ ]:
import requests
import time
import json
import re
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from openai import OpenAI

print("Librerías cargadas correctamente")

In [ ]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

CAMPOS_OFERTA = [
    "marca",
    "modelo",
    "precio_vehiculo",
    "cuota_mensual",
    "plazo_meses",
    "entrada",
    "tin",
    "tae",
    "comision_apertura",
    "valor_residual",
    "importe_financiado",
    "tipo_financiacion",
    "fecha_fin_oferta",
    "url",
    "fecha_extraccion"
]

print(f"API Key OpenAI cargada: {'OK' if OPENAI_API_KEY else 'ERROR — revisa los Secrets'}")

## 3. Funciones de scraping

In [ ]:
def scrape_estatico(url, reintentos=3, pausa=2):
    """Descarga una página estática con requests. Reintenta si falla."""
    for intento in range(reintentos):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            print(f"  [intento {intento+1}/{reintentos}] Error en {url}: {e}")
            time.sleep(pausa * (intento + 1))
    return None


def crear_driver():
    """Crea un driver de Chrome headless. Usa webdriver-manager para encontrar el driver correcto."""
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(f"user-agent={HEADERS['User-Agent']}")
    driver_path = ChromeDriverManager().install()
    return webdriver.Chrome(service=Service(driver_path), options=options)


def scroll_hasta_el_final(driver, pausas=5):
    """Hace scroll progresivo para forzar la carga de contenido lazy."""
    altura_total = driver.execute_script("return document.body.scrollHeight")
    paso = altura_total // pausas
    for i in range(1, pausas + 1):
        driver.execute_script(f"window.scrollTo(0, {paso * i});")
        time.sleep(1)
    driver.execute_script("window.scrollTo(0, 0);")


def scrape_dinamico(url, espera_extra=3, scroll=False):
    """Descarga una página que requiere JavaScript con Selenium."""
    driver = crear_driver()
    try:
        driver.get(url)
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(espera_extra)
        if scroll:
            scroll_hasta_el_final(driver)
            time.sleep(2)
        return driver.page_source
    except Exception as e:
        print(f"  Error Selenium en {url}: {e}")
        return None
    finally:
        driver.quit()


PALABRAS_CLAVE = ["TIN", "TAE", "cuota", "€/mes", "financiaci", "entrada", "plazo",
                  "apertura", "interés", "mensual", "residual", "financiado"]

def html_a_texto(html, ventana=3000):
    """
    Convierte HTML a texto limpio y extrae solo los fragmentos con datos financieros.
    Reduce el consumo de tokens un ~80% respecto a enviar la página entera.
    """
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "noscript"]):
        tag.decompose()
    texto = soup.get_text(separator=" ", strip=True)
    texto = re.sub(r'\s+', ' ', texto)

    # Buscar posición de la primera palabra clave financiera
    pos = -1
    for palabra in PALABRAS_CLAVE:
        idx = texto.lower().find(palabra.lower())
        if idx != -1 and (pos == -1 or idx < pos):
            pos = idx

    if pos != -1:
        # Extraer ventana alrededor del área financiera
        inicio = max(0, pos - 200)
        fin = min(len(texto), pos + ventana)
        return texto[inicio:fin]

    # Si no se encuentran palabras clave, devolver el inicio del texto
    return texto[:ventana]


print("Funciones de scraping definidas")

## 4. Extracción con LLM (Claude)

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

PROMPT_SISTEMA = """Eres un experto en análisis de ofertas de financiación de automóviles en España.
Tu tarea es extraer información estructurada de textos de páginas web de concesionarios.
Devuelve SIEMPRE un JSON válido con los campos indicados.
Si un campo no aparece en el texto, devuelve null para ese campo.
No inventes datos. Solo extrae lo que esté explícitamente en el texto."""


def extraer_oferta_con_llm(texto, marca, url):
    """
    Envía el texto de la página a GPT-4o-mini y extrae los campos de la oferta.
    Devuelve una lista de dicts (puede haber varias ofertas en una misma página).
    """
    prompt = f"""Analiza el siguiente texto extraído de la web de {marca} ({url}) y extrae TODAS las ofertas de financiación que encuentres.

Para cada oferta, devuelve un objeto JSON con estos campos:
- modelo: nombre del modelo de coche
- precio_vehiculo: precio total del vehículo (número, sin símbolo €)
- cuota_mensual: cuota mensual en euros (número)
- plazo_meses: duración en meses (número)
- entrada: entrada inicial en euros (número, 0 si no se requiere)
- tin: Tasa de Interés Nominal en % (número)
- tae: Tasa Anual Equivalente en % (número)
- comision_apertura: comisión de apertura en euros (número, 0 si es gratuita)
- valor_residual: valor residual o valor futuro garantizado en euros (número o null)
- importe_financiado: capital financiado en euros (número)
- tipo_financiacion: tipo de producto ("credito", "leasing", "renting", "PCP", u otro)
- fecha_fin_oferta: fecha límite de la oferta en formato YYYY-MM-DD (string o null)

Devuelve SOLO un JSON con esta estructura, sin texto adicional:
{{"ofertas": [ {{...}}, {{...}} ]}}

TEXTO DE LA PÁGINA:
{texto}"""

    try:
        respuesta = client.chat.completions.create(
            model="gpt-4o-mini",
            max_tokens=2048,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": PROMPT_SISTEMA},
                {"role": "user", "content": prompt}
            ]
        )
        texto_respuesta = respuesta.choices[0].message.content
        datos = json.loads(texto_respuesta)
        ofertas = datos.get("ofertas", [])
        for oferta in ofertas:
            oferta["marca"] = marca
            oferta["url"] = url
            oferta["fecha_extraccion"] = str(date.today())
        return ofertas
    except Exception as e:
        print(f"  Error LLM para {url}: {e}")
        return []


print("Cliente OpenAI (gpt-4o-mini) configurado")

## 5. Pipeline completo: scraping + extracción LLM

In [ ]:
def procesar_url(url, marca, usar_selenium=False, scroll=False):
    """Pipeline completo para una URL: descarga → texto limpio → LLM → datos estructurados."""
    print(f"Procesando: {marca} — {url}")

    if usar_selenium:
        html = scrape_dinamico(url, scroll=scroll)
    else:
        html = scrape_estatico(url)

    if not html:
        print(f"  No se pudo descargar {url}")
        return []

    texto = html_a_texto(html)
    print(f"  Texto extraído: {len(texto)} caracteres")

    ofertas = extraer_oferta_con_llm(texto, marca, url)
    print(f"  Ofertas encontradas: {len(ofertas)}")
    return ofertas


print("Pipeline definido")

## 6. URLs de la competencia

In [ ]:
COMPETENCIA = {
    "TOYOTA": {
        "selenium": False,
        "scroll": False,
        "urls": [
            "https://www.toyota.es/promociones/toyota-c-hr-plus-easy-plus",
            "https://www.toyota.es/promociones/toyota-c-hr-140h-advance-easy-plus",
            "https://www.toyota.es/promociones/toyota-c-hr-plug-in-hybrid-220ph-advance-easy-plus",
            "https://www.toyota.es/promociones/yaris-ng-active-tech-easy-plus",
            "https://www.toyota.es/promociones/yaris-cross-hybrid-style-easy-plus",
            "https://www.toyota.es/promociones/corolla-hybrid-140h-active-plus-easy-plus",
            "https://www.toyota.es/promociones/corolla-sedan-hybrid-140h-style-plus-easy-plus",
            "https://www.toyota.es/promociones/corolla-touring-sports-hybrid-140h-easy-plus",
            "https://www.toyota.es/promociones/corolla-cross-hybrid-style-easy-plus",
            "https://www.toyota.es/promociones/toyota-bz4x-electric-4x2-advance-easy-plus",
            "https://www.toyota.es/promociones/aygo-x-cross-play-easy",
            "https://www.toyota.es/promociones/rav4-hybrid-220h-2x4-advance-easy-plus",
            "https://www.toyota.es/promociones/rav4-plug-in-hybrid-300ph-advance-easy-plus"
        ]
    },
    "VOLKSWAGEN": {
        "selenium": True,
        "scroll": True,
        "urls": [
            "https://www.volkswagen.es/es/ofertas.html"
        ]
    },
    "PEUGEOT": {
        "selenium": True,
        "scroll": True,
        "urls": [
            "https://www.peugeot.es/comprar/ofertas-del-momento.html"
        ]
    },
    "RENAULT": {
        "selenium": False,
        "scroll": False,
        "urls": [
            "https://promociones.renault.es/particulares/clio/",
            "https://promociones.renault.es/particulares/captur/",
            "https://promociones.renault.es/particulares/symbioz/",
            "https://promociones.renault.es/particulares/symbioz-glp/",
            "https://promociones.renault.es/particulares/austral/",
            "https://promociones.renault.es/particulares/arkana/",
            "https://promociones.renault.es/particulares/espace/",
            "https://promociones.renault.es/particulares/rafale/",
            "https://promociones.renault.es/particulares/rafale-phev/"
        ]
    },
    "NISSAN": {
        "selenium": True,
        "scroll": True,
        "urls": [
            "https://www.nissan.es/vehiculos/ofertas.html"
        ]
    },
    "HYUNDAI": {
        "selenium": True,
        "scroll": True,
        "urls": [
            "https://www.hyundai.com/es/es/planenchufados.html",
            "https://www.hyundai.com/es/es/modelos/inster.html",
            "https://www.hyundai.com/es/es/modelos/tucson/blackline.html",
            "https://www.hyundai.com/es/es/suv.html"
        ]
    },
    "AUDI": {
        "selenium": True,
        "scroll": True,
        "urls": [
            "https://www.audi.es/es/compra/promociones/"
        ]
    }
}

print(f"Configuradas {sum(len(v['urls']) for v in COMPETENCIA.values())} URLs de {len(COMPETENCIA)} marcas")

## 7. Ejecución del scraping

In [ ]:
# Ejecutar scraping completo de todas las marcas
# Para probar con una sola marca: MARCAS_A_EJECUTAR = ["TOYOTA"]
MARCAS_A_EJECUTAR = list(COMPETENCIA.keys())

todas_las_ofertas = []

for marca in MARCAS_A_EJECUTAR:
    config = COMPETENCIA[marca]
    print(f"\n{'='*50}")
    print(f"MARCA: {marca}")
    print(f"{'='*50}")

    for url in config["urls"]:
        ofertas = procesar_url(url, marca, usar_selenium=config["selenium"])
        todas_las_ofertas.extend(ofertas)
        time.sleep(2)  # Pausa educada entre peticiones

print(f"\nTotal de ofertas extraídas: {len(todas_las_ofertas)}")

MARCAS_A_EJECUTAR = list(COMPETENCIA.keys())

todas_las_ofertas = []

for marca in MARCAS_A_EJECUTAR:
    config = COMPETENCIA[marca]
    print(f"\n{'='*50}")
    print(f"MARCA: {marca}")
    print(f"{'='*50}")

    for url in config["urls"]:
        ofertas = procesar_url(
            url, marca,
            usar_selenium=config["selenium"],
            scroll=config.get("scroll", False)
        )
        if not ofertas:
            print(f"  AVISO: Sin datos para {url}")
        todas_las_ofertas.extend(ofertas)
        time.sleep(2)

print(f"\n{'='*50}")
print(f"RESUMEN: {len(todas_las_ofertas)} ofertas extraídas en total")
marcas_con_datos = set(o["marca"] for o in todas_las_ofertas)
marcas_sin_datos = set(MARCAS_A_EJECUTAR) - marcas_con_datos
if marcas_sin_datos:
    print(f"Marcas SIN datos: {marcas_sin_datos}")
print(f"Marcas CON datos: {marcas_con_datos}")

In [ ]:
# Crear DataFrame con todas las ofertas
df = pd.DataFrame(todas_las_ofertas)

if df.empty:
    print("No se han extraído ofertas. Revisa los mensajes de error arriba.")
else:
    # marca y modelo siempre primero, luego el resto de campos
    cols_ordenadas = ["marca", "modelo"] + [c for c in CAMPOS_OFERTA if c not in ("marca", "modelo") and c in df.columns]
    df = df[cols_ordenadas]

    pd.set_option("display.max_columns", None)
    pd.set_option("display.max_rows", 50)

    print(f"Total ofertas extraídas: {len(df)}")
    print(f"Marcas: {sorted(df['marca'].unique().tolist())}")
    print(f"Modelos por marca:\n{df.groupby('marca')['modelo'].count().to_string()}")
    display(df)

In [ ]:
# Exportar a CSV
nombre_archivo = f"ofertas_competencia_{date.today().strftime('%Y%m%d')}.csv"
df.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")
print(f"Guardado en: {nombre_archivo}")

# En Colab, descargar el archivo:
# from google.colab import files
# files.download(nombre_archivo)

In [ ]:
# Resumen comparativo por marca
if not df.empty and "marca" in df.columns:
    resumen = df.groupby("marca").agg(
        num_ofertas=("modelo", "count"),
        tin_medio=("tin", "mean"),
        tae_medio=("tae", "mean"),
        cuota_min=("cuota_mensual", "min"),
        cuota_max=("cuota_mensual", "max")
    ).round(2)
    print(resumen)